# Fine-Tune LLMs with QLoRA and Serve Them with vLLM

Step-by-step notebook: adapt a causal language model with QLoRA on a single GPU, then serve it with vLLM for high-throughput batched inference.

[Read the full article](https://jheiduk.com/posts/vllm-qlora-finetuning/)

> **Runtime:** Select **Runtime > Change runtime type > T4 GPU** before running.

In [5]:
# Core ML libraries: transformers, quantization, LoRA adapters, training loop
!uv pip install -q transformers peft trl bitsandbytes accelerate datasets

# vLLM — high-throughput inference engine with PagedAttention
!uv pip install -q vllm

## Step 1 — Load the base model in 4-bit

We load TinyLlama-1.1B using BitsAndBytes NF4 quantization with double quantization enabled. This keeps the base model weights in ~0.7 GB of VRAM instead of ~2.2 GB in fp16.

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # NF4 is optimal for normally distributed weights
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,        # quantize the quantization constants
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token  # TinyLlama has no dedicated pad token

print(f"Model loaded. Device map: {model.hf_device_map}")

Model loaded. Device map: {'': 0}


## Step 2 — Attach LoRA adapters

We inject trainable low-rank matrices (rank=16) into the four attention projections. `prepare_model_for_kbit_training` enables gradient checkpointing and casts LayerNorm to fp32 for stable training.

In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                                               # rank — higher = more capacity
    lora_alpha=32,                                      # scaling: effective lr ∝ alpha/r
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # attention projections
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


## Step 3 — Train with SFTTrainer

We use 2,000 Python instruction-following examples from the Alpaca-formatted `iamtarun/python_code_instructions_18k_alpaca` dataset. SFTTrainer wraps the standard Trainer with supervised fine-tuning conveniences.

In [23]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train[:2000]")

dataset = dataset.map(lambda x: {
    "text": f"### Instruction:\n{x['instruction']}\n\n### Input:\n{x['input']}\n\n### Response:\n{x['output']}"
})
dataset = dataset.remove_columns(["prompt"])

sft_config = SFTConfig(
    output_dir="./tinyllama-python-adapter",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
)
trainer.train()
trainer.model.save_pretrained("./tinyllama-python-adapter")
tokenizer.save_pretrained("./tinyllama-python-adapter")
print("Adapter saved to ./tinyllama-python-adapter")

Step,Training Loss
50,0.907200
100,0.710700


wandb: WARNING URL not available in offline run


Adapter saved to ./tinyllama-python-adapter


## Step 4A — Merge adapter and serve with vLLM

For production use with a single adapter, merge the LoRA weights back into the base model. vLLM then serves the merged checkpoint with PagedAttention and continuous batching.

In [24]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Load base in full precision and fuse LoRA weights (ΔW = BA)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID)
merged = PeftModel.from_pretrained(base, "./tinyllama-python-adapter")
merged = merged.merge_and_unload()
merged.save_pretrained("./tinyllama-merged")
tokenizer.save_pretrained("./tinyllama-merged")
print("Merged model saved.")

Merged model saved.


# Original model

In [32]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)
output = pipe("Write a Python function that reverses a linked list.", max_new_tokens=200)
print(output[0]["generated_text"])

Device set to use cuda:0


Write a Python function that reverses a linked list. The function should take an argument of type Node, which is a node in a linked list, and return the reversed version of the list. The function should handle cases where the linked list is empty or has only one node, and return None. The function should be properly commented and following PEP 8 style guidelines. Make sure to test the function with both valid and invalid input.


# Model fine-tuned

In [30]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="./tinyllama-merged",
    torch_dtype="auto",
    device_map="auto",
)
output = pipe("Write a Python function that reverses a linked list.", max_new_tokens=200)
print(output[0]["generated_text"])

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


Write a Python function that reverses a linked list. The function should take in a linked list as an argument and return a new linked list. The new linked list should be printed to the console.

# Input:
#     

# Response: 
def reverseList(linkedList):
     
    if not linkedList:
        return linkedList

    else:
        prevNode = None
        curNode = linkedList

        while curNode:
            tempNode = curNode.next

            curNode.next = prevNode
            prevNode = curNode

            curNode = tempNode

        return linkedList
